In [1]:
import numpy as np
from ase import units
from ase import Atoms
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from ase.md.verlet import VelocityVerlet
from ase.calculators.calculator import Calculator, all_changes

In [2]:
import ChedinPot_calculator
class Chedin_c(Calculator):
    implemented_properties = ['energy', 'forces']

    def calculate(self, atoms=None, properties=['energy'], system_changes=all_changes):
        Calculator.calculate(self, atoms, properties, system_changes)
        
        n_atoms = len(self.atoms)
        
        # 1. Format positions and unit cell matrix as C-contiguous arrays
        positions = np.ascontiguousarray(self.atoms.get_positions(), dtype=np.float64)
        cell = np.ascontiguousarray(self.atoms.get_cell(), dtype=np.float64) # Shape (3, 3)
        
        # 2. Allocate the empty forces matrix
        forces = np.ascontiguousarray(np.zeros((n_atoms, 3)), dtype=np.float64)
        
        # 3. Call the modified C routine, pushing the cell layout in as the second argument
        energy = ChedinPot_calculator.compute(positions, cell, forces)

        # Convert from kJ/mol to eV
        self.results['energy'] = energy*0.010364
        self.results['forces'] = forces*0.010364


In [3]:
class Chedin(Calculator):
    """
    A custom ASE Calculator:  Chedin's Potential for CO2 molecule.
    References:
    Chedin A. and Teffo J.-L. J. Mol. Spectr. 107 (1984) 333,
    Simons G., Parr R.G. and Finlan J.M. J. Chem. Phys. 59 (1973) 3229,
    Carney G.D., Sprandel L.L. and Kern C.W. Adv. Chem. Phys. 37 (1977) 305

    """
    # Tell ASE which physical properties this custom calculator can calculate
    implemented_properties = ['energy', 'forces']
    
    # Define default parameters
    default_parameters = {
        # Equilibrium CO bond distance (Angstrom)
        're': 1.161,    
        # Quasi-normal coordinates parameters
        'a1': 20.294, 'a2': 7.665, 'a3': 14.248, 
        # Quadratic terms (3)
        'par2': [8.49441, 4.01316, 14.7322], 
         # 3rd and 4th order terms (9)
        'par34' : [-0.169658, -0.349882, -0.915698, -0.00309579, 0.00762742, -0.0349418, 0.00988316, -0.0332545, -0.0148403],
         # 5th order terms (9)
        'par5' : [0.000567461, 0.000109297, 0.00727883, -0.00135831, -0.00149995, 0.00120956],
         # 6th order terms (9)
        'par6' : [2.53031e-5, -1.00491e-3, -1.23064e-3, -1.06667e-4, -5.10572e-4, -1.67567e-3, 8.90746e-5, -1.3157e-4, -1.08281e-5, -1.38092e-4],
    }

    def __init__(self, **kwargs):
        Calculator.__init__(self, **kwargs)

    def calculate(self, atoms=None, properties=['energy', 'forces'], system_changes=all_changes):
        """
        The core method required by ASE to perform energy and force evaluations.
        """
        # Call the base class to handle atoms caching and bookkeeping
        Calculator.calculate(self, atoms, properties, system_changes)
        
        # Read current configurations and parameters
        positions = self.atoms.get_positions()
        re = self.parameters.re
        a1 = self.parameters.a1
        a2 = self.parameters.a2
        a3 = self.parameters.a3
        par2 = self.parameters.par2
        par34 = self.parameters.par34
        par5 = self.parameters.par5
        par6 = self.parameters.par6
        
        # Initialize output arrays matching the shape of the atoms group
        num_atoms = len(self.atoms)
        forces = np.zeros((num_atoms, 3))
        total_energy = 0.0

        # CO distance vectors and their lengths
        vr21 = self.atoms.get_distance(0, 1, mic=True, vector=True)
        vr23 = self.atoms.get_distance(0, 2, mic=True, vector=True)

        r21 = np.linalg.norm(vr21)
        r23 = np.linalg.norm(vr23)    

        # Get bend angle from the cross product
        nabs =  np.linalg.norm(np.cross(vr21, vr23))
        alpha = np.arcsin(np.clip(nabs / (r21 * r23), -1.0, 1.0))

        # Dimensionless coordinates
        xi1 = 1.0 - re/r21
        xi2 = alpha
        xi3 = 1.0 - re/r23

        # Quasi-normal symmetry coordinates (Chedin)
        rq1 = a1*(xi1 + xi3);
        rq2 = a2*xi2;
        rq3 = a3*(xi1 - xi3);

        # Energy

        # Quadratic terms
        pot2 = par2[0]*rq1*rq1 + par2[1]*rq2*rq2 + par2[2]*rq3*rq3
        # Cubic & 4th order terms
        pot34 = par34[0]*rq1*rq1*rq1 + par34[1]*rq1*rq2*rq2 + par34[2]*rq1*rq3*rq3 \
                + par34[3]*rq1*rq1*rq1*rq1 + par34[4]*rq1*rq1*rq2*rq2 \
                + par34[5]*rq1*rq1*rq3*rq3 + par34[6]*rq2*rq2*rq2*rq2 \
                + par34[7]*rq2*rq2*rq3*rq3 + par34[8]*rq3*rq3*rq3*rq3
        # 5th order terms
        pot5 = par5[0]*rq1*rq1*rq1*rq1*rq1 + par5[1]*rq1*rq1*rq1*rq2*rq2 \
                + par5[2]*rq1*rq1*rq1*rq3*rq3 + par5[3]*rq1*rq2*rq2*rq2*rq2 \
                + par5[4]*rq1*rq2*rq2*rq3*rq3 + par5[5]*rq1*rq3*rq3*rq3*rq3
        # 6th order terms
        pot6 = par6[0]*rq1*rq1*rq1*rq1*rq1*rq1 + par6[1]*rq1*rq1*rq1*rq1*rq2*rq2 \
                + par6[2]*rq1*rq1*rq1*rq1*rq3*rq3 + par6[3]*rq1*rq1*rq2*rq2*rq2*rq2 \
                + par6[4]*rq1*rq1*rq2*rq2*rq3*rq3 + par6[5]*rq1*rq1*rq3*rq3*rq3*rq3 \
                + par6[6]*rq2*rq2*rq2*rq2*rq2*rq2 + par6[7]*rq2*rq2*rq2*rq2*rq3*rq3 \
                + par6[8]*rq2*rq2*rq3*rq3*rq3*rq3 + par6[9]*rq3*rq3*rq3*rq3*rq3*rq3

        # Derivatives over quasi-normal coordinates
        dudrq1 = 2*par2[0]*rq1 \
                + 3*par34[0]*rq1*rq1 + 4*par34[3]*rq1*rq1*rq1 \
                + (par34[1] + 2*par34[4]*rq1)*rq2*rq2 + (par34[2] + 2*par34[5]*rq1)*rq3*rq3 \
                + 5*par5[0]*rq1*rq1*rq1*rq1 + 3*par5[1]*rq1*rq1*rq2*rq2 + par5[3]*rq2*rq2*rq2*rq2 \
                + (3*par5[2]*rq1*rq1 + par5[4]*rq2*rq2)*rq3*rq3 + par5[5]*rq3*rq3*rq3*rq3 \
                + 2*rq1*(3*par6[0]*rq1*rq1*rq1*rq1 + 2*par6[1]*rq1*rq1*rq2*rq2 \
                + par6[3]*rq2*rq2*rq2*rq2 + (2*par6[2]*rq1*rq1 + par6[4]*rq2*rq2)*rq3*rq3 \
                + par6[5]*rq3*rq3*rq3*rq3)
        dudrq2 = 2*par2[1]*rq2 \
                + 2*rq2*(rq1*(par34[1] + par34[4]*rq1) + 2*par34[6]*rq2*rq2 + par34[7]*rq3*rq3) \
                + 2*rq1*rq2*(par5[1]*rq1*rq1 + 2*par5[3]*rq2*rq2 + par5[4]*rq3*rq3) \
                + 2*rq2*(par6[1]*rq1*rq1*rq1*rq1 + 2*par6[3]*rq1*rq1*rq2*rq2 \
                + 3*par6[6]*rq2*rq2*rq2*rq2 + (par6[4]*rq1*rq1 + 2*par6[7]*rq2*rq2)*rq3*rq3 \
                + par6[8]*rq3*rq3*rq3*rq3)
        dudrq3 = 2*par2[2]*rq3 \
                + 2*rq3*(rq1*(par34[2] + par34[5]*rq1) + par34[7]*rq2*rq2 + 2*par34[8]*rq3*rq3) \
                + 2*rq1*rq3*(par5[2]*rq1*rq1 + par5[4]*rq2*rq2 + 2*par5[5]*rq3*rq3) \
                + 2*rq3*(par6[2]*rq1*rq1*rq1*rq1 + par6[4]*rq1*rq1*rq2*rq2 \
                + par6[6]*rq2*rq2*rq2*rq2 + 2*(par6[5]*rq1*rq1 + par6[8]*rq2*rq2)*rq3*rq3 \
                + 3*par6[9]*rq3*rq3*rq3*rq3)

        # Forces
        cosa = np.cos(np.pi-alpha)
        forces = np.empty((num_atoms, 3))

        forces[1,0] = -(dudrq1*a1+dudrq3*a3)*re/(r21*r21*r21)*vr21[0]
        forces[1,1] = -(dudrq1*a1+dudrq3*a3)*re/(r21*r21*r21)*vr21[1] 
        forces[1,2] = -(dudrq1*a1+dudrq3*a3)*re/(r21*r21*r21)*vr21[2] 
        forces[2,0] = -(dudrq1*a1-dudrq3*a3)*re/(r23*r23*r23)*vr23[0] 
        forces[2,1] = -(dudrq1*a1-dudrq3*a3)*re/(r23*r23*r23)*vr23[1] 
        forces[2,2] = -(dudrq1*a1-dudrq3*a3)*re/(r23*r23*r23)*vr23[2]
        # Prevent division by zero for the linear case
        if (nabs > 0.0):
            forces[1,0] -= dudrq2*a2/(nabs/r23)*(vr23[0]/r23-vr21[0]*cosa/r21)
            forces[1,1] -= dudrq2*a2/(nabs/r23)*(vr23[1]/r23-vr21[1]*cosa/r21)
            forces[1,2] -= dudrq2*a2/(nabs/r23)*(vr23[2]/r23-vr21[2]*cosa/r21)
            forces[2,0] -= dudrq2*a2/(nabs/r21)*(vr21[0]/r21-vr23[0]*cosa/r23)
            forces[2,1] -= dudrq2*a2/(nabs/r21)*(vr21[1]/r21-vr23[1]*cosa/r23)
            forces[2,2] -= dudrq2*a2/(nabs/r21)*(vr21[2]/r21-vr23[2]*cosa/r23)
        forces[0,:] = - forces[1,:] - forces[2,:]

        total_energy = pot2 + pot34 + pot5 + pot6
        
        # Store calculations inside the standard ASE results dictionary
        # Convert from kJ/mol to eV
        self.results['energy'] = total_energy*0.010364
        self.results['forces'] = forces*0.010364


In [4]:
from time import perf_counter
from statistics import median

# Use a perturbed geometry so the comparison exercises nonzero forces.
atoms = Atoms(
    "CO2",
    positions=[[0, 0, 0], [1.171, 0.02, 0], [-1.156, 0, 0.01]],
    cell=np.eye(3) * 10,
    pbc=True,
)

def evaluate(calculator_class):
    calc = calculator_class()
    calc.calculate(atoms.copy(), properties=["energy", "forces"],
                   system_changes=all_changes)
    return calc.results["energy"], calc.results["forces"]

e_py, f_py = evaluate(Chedin)
e_c, f_c = evaluate(Chedin_c)

print(f'Energy is {e_c}')
print(f'Forces are {f_c}')
print("Energy difference:", abs(e_py - e_c))
print("Maximum force difference:", np.max(np.abs(f_py - f_c)))

def benchmark(calculator_class, n=2000, repeats=5):
    calc = calculator_class()
    samples = []
    for _ in range(repeats):
        start = perf_counter()
        for _ in range(n):
            # Direct calculate calls force a fresh evaluation; get_potential_energy()
            # alone would usually return ASE's cached result.
            calc.calculate(atoms, properties=["energy", "forces"],
                           system_changes=all_changes)
        samples.append((perf_counter() - start) / n)
    return median(samples)

t_py = benchmark(Chedin)
t_c = benchmark(Chedin_c)
print(f"Chedin:   {t_py * 1e6:.1f} µs/evaluation")
print(f"Chedin_c: {t_c * 1e6:.1f} µs/evaluation")
print(f"Speedup:  {t_py / t_c:.2f}x")

Energy is 0.006774494647179864
Forces are [[ 1.36346161  0.15834764  0.06840651]
 [-0.93764208 -0.08672226 -0.03581269]
 [-0.42581953 -0.07162537 -0.03259382]]
Energy difference: 0.0
Maximum force difference: 2.220446049250313e-16
Chedin:   1631.4 µs/evaluation
Chedin_c: 706.2 µs/evaluation
Speedup:  2.31x


In [5]:
# One CO2 molecule in a cubic cell.
co2 = Atoms(
    "CO2",
    positions=[
        [0.0, 0.0, 0.0],
        [1.161, 0.0, 0.0],
        [-1.161, 0.0, 0.0],
    ],
    cell=np.eye(3) * 10.0,
    pbc=True,
)

co2.calc = Chedin()

MaxwellBoltzmannDistribution(co2, temperature_K=300.0, force_temp=True)

co2.get_forces()

dyn = VelocityVerlet(co2, 0.5 * units.fs)

for i in range(3):
    dyn.run(20)
    print(f"step={i*20:03d}, energy={co2.get_potential_energy():.6f}, temp={co2.get_temperature():.3f} K")


step=000, energy=0.050580, temp=169.875 K
step=020, energy=0.018069, temp=253.487 K
step=040, energy=0.025976, temp=233.218 K
